# Two materials, one stiffness: micromechanics, step by step

This notebook follows the *Learn* part of the lesson
[Two materials, one stiffness](https://m-aryayi.github.io/composites-playground/tools/rve-homogenization/)
of Composites Playground. Each step has a short text and its code.
Run the cells from the top to the bottom. Then change the materials or Vf and run them again.

Units: GPa for moduli, g/cm³ for densities. Axis 1 is along the fibers.
References: Kaw (2005), *Mechanics of Composite Materials*, 2nd ed.;
Chawla (2012), *Composite Materials*, 3rd ed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Fiber and matrix

A carbon fiber is transversely isotropic: five constants E1, E2, ν12, ν23, G12
(Kaw). The epoxy matrix is isotropic: E and ν (Kaw).

In [ ]:
fiber = {"e1": 230.0, "e2": 22.0, "nu12": 0.30, "nu23": 0.35, "g12": 22.0}
em, num = 3.4, 0.30            # epoxy: E (GPa) and ν
rho_f, rho_m = 1.80, 1.20      # g/cm³
vf = 0.6                       # fiber volume fraction


def shear_modulus(e, nu):
    """Return the shear modulus of an isotropic material."""
    return e / (2.0 * (1.0 + nu))


gm = shear_modulus(em, num)
gf23 = shear_modulus(fiber["e2"], fiber["nu23"])   # the 2-3 plane is isotropic
print(f"Gm = {gm:.3f} GPa,  fiber G23 = {gf23:.2f} GPa")

## 2. Fiber volume fraction and density

In a hexagonal pattern the fibers touch at Vf = π / (2√3). The density is a
simple mix.

In [ ]:
print(f"largest Vf of a hexagonal pattern: {np.pi / (2 * np.sqrt(3)):.3f}")
rho_c = rho_f * vf + rho_m * (1 - vf)
print(f"composite density at Vf = {vf}: {rho_c:.2f} g/cm³")

## 3. The rule of mixtures

E1 and ν12: parallel springs. G12: springs in a row.
E2: an improved form with √Vf. No G23, no ν23.

In [ ]:
def rule_of_mixtures(fiber, em, num, vf):
    """Return E1, E2, G12, G23, ν12, ν23 by the rule of mixtures (NaN: no value)."""
    vm = 1.0 - vf
    gm = shear_modulus(em, num)
    root = np.sqrt(vf)
    e1 = vf * fiber["e1"] + vm * em
    e2 = 1.0 / (root / (fiber["e2"] * root + em * (1.0 - root)) + (1.0 - root) / em)
    g12 = fiber["g12"] * gm / (gm * vf + fiber["g12"] * vm)
    nu12 = vf * fiber["nu12"] + vm * num
    return e1, e2, g12, np.nan, nu12, np.nan


print([round(float(v), 3) for v in rule_of_mixtures(fiber, em, num, vf)])

Compare with the book: glass (E = 85 GPa, ν = 0.2) in epoxy at Vf = 0.70.
Kaw gives E1 = 60.52 GPa with the rule of mixtures.

In [ ]:
glass = {"e1": 85.0, "e2": 85.0, "nu12": 0.20, "nu23": 0.20,
         "g12": shear_modulus(85.0, 0.20)}
print(f"E1 = {rule_of_mixtures(glass, 3.4, 0.30, 0.70)[0]:.2f} GPa")

## 4. Chamis

Chamis uses √Vf for the properties across the fibers.

In [ ]:
def chamis(fiber, em, num, vf):
    """Return E1, E2, G12, G23, ν12, ν23 by Chamis."""
    vm = 1.0 - vf
    gm = shear_modulus(em, num)
    root = np.sqrt(vf)
    e1 = vf * fiber["e1"] + vm * em
    e2 = em / (1.0 - root * (1.0 - em / fiber["e2"]))
    g12 = gm / (1.0 - root * (1.0 - gm / fiber["g12"]))
    g23 = gm / (1.0 - root * (1.0 - gm / shear_modulus(fiber["e2"], fiber["nu23"])))
    nu12 = vf * fiber["nu12"] + vm * num
    nu23 = e2 / (2.0 * g23) - 1.0
    return e1, e2, g12, g23, nu12, nu23

## 5. Composite cylinders (CCA)

Each fiber sits in a ring of matrix; a three-phase model gives G23.
The fiber enters with its own plane-strain bulk modulus
K23 = E2 / (2 (1 − ν23 − 2 ν12² E2 / E1)).

In [ ]:
def cca(fiber, em, num, vf):
    """Return E1, E2, G12, G23, ν12, ν23 by the composite cylinder assemblage."""
    vm = 1.0 - vf
    gm = shear_modulus(em, num)
    gf12, gf23, nuf = fiber["g12"], shear_modulus(fiber["e2"], fiber["nu23"]), fiber["nu12"]
    kf = fiber["e2"] / (2.0 * (1.0 - fiber["nu23"]
                               - 2.0 * fiber["nu12"] ** 2 * fiber["e2"] / fiber["e1"]))
    km = em / (2.0 * (1.0 - 2.0 * num) * (1.0 + num))
    mix = vf / km + 1.0 / gm + vm / kf
    e1 = vf * fiber["e1"] + vm * em + 4.0 * vf * vm * (nuf - num) ** 2 / mix
    nu12 = vf * nuf + vm * num + vf * vm * (nuf - num) * (1.0 / km - 1.0 / kf) / mix
    g12 = gm * (gf12 * (1.0 + vf) + gm * vm) / (gf12 * vm + gm * (1.0 + vf))
    eta_m, eta_f, mu = 3.0 - 4.0 * num, 1.0 + 2.0 * gf23 / kf, gf23 / gm
    common = 3.0 * vf * vm ** 2 * (mu - 1.0) * (mu + eta_f)
    cube = (mu * eta_m - eta_f) * vf ** 3
    a = (common + (mu * eta_m + eta_f * eta_m - cube)
         * (eta_m * vf * (mu - 1.0) - (mu * eta_m + 1.0)))
    b = (-common + 0.5 * (mu * eta_m + 1.0 + (mu - 1.0) * vf)
         * ((eta_m - 1.0) * (mu + eta_f) - 2.0 * cube)
         + 0.5 * (eta_m + 1.0) * vf * (mu - 1.0) * (mu + eta_f + cube))
    c = common + (mu * eta_m + (mu - 1.0) * vf + 1.0) * (mu + eta_f + cube)
    g23 = gm * (-b - np.sqrt(b ** 2 - a * c)) / a
    k23 = ((km * (kf + gm) * vm + kf * (km + gm) * vf)
           / ((kf + gm) * vm + (km + gm) * vf))
    m = 1.0 + 4.0 * k23 * nu12 ** 2 / e1
    nu23 = (k23 - m * g23) / (k23 + m * g23)
    e2 = 2.0 * (1.0 + nu23) * g23
    return e1, e2, g12, g23, nu12, nu23

## 6. Mori-Tanaka

Each fiber feels the average strain of the matrix. For a unidirectional
ply this gives closed formulas.

In [ ]:
def mori_tanaka(fiber, em, num, vf):
    """Return E1, E2, G12, G23, ν12, ν23 by Mori-Tanaka."""
    vm = 1.0 - vf
    ef1, ef2, nuf12, nuf23, gf12 = (fiber[k] for k in ("e1", "e2", "nu12", "nu23", "g12"))
    z1 = 1.0 / (-2.0 * vm * nuf12 ** 2 / ef1 + vm * (1.0 - nuf23) / ef2
                + (1.0 + num) * (1.0 + vf * (1.0 - 2.0 * num)) / em)
    z2 = ef2 * (3.0 + vf - 4.0 * num) * (1.0 + num) + vm * em * (1.0 + nuf23)
    e1 = vf * ef1 + vm * em + 2.0 * vf * vm * z1 * (nuf12 - num) ** 2
    e2 = e1 / ((1.0 - num ** 2) * (
        1.0 / (1.0 - num ** 2)
        + 2.0 * vf * e1 * (1.0 + nuf23 - ef2 * (1.0 + num) / em) / z2
        + vf * z1 * ef1 * ((1.0 + num) / em - 2.0 / ef1 + (1.0 - nuf23) / ef2) / em))
    nu12 = num + 2.0 * vf * z1 * (nuf12 - num) * (1.0 - num ** 2) / em
    inner = 1.0 - 2.0 * z1 * (vf * ef1 * (1.0 - num ** 2)
                              + vm * em * (1.0 - nuf12 ** 2)) / (ef1 * em)
    nu23 = 1.0 - e2 * (2.0 + 2.0 * vf * ef1 * (1.0 - num ** 2) / (vm * em) * inner
                       - e1 * (1.0 + num) / em) / e1
    g12 = (em * (1.0 + vf - 4.0 * vf / (1.0 + vf + 2.0 * vm * gf12 * (1.0 + num) / em))
           / (2.0 * vm * (1.0 + num)))
    d = em / shear_modulus(ef2, nuf23) - 2.0 * (1.0 + num)
    g23 = em / (2.0 * (1.0 + num) + vf * d / (1.0 + vm * d / (8.0 * (1.0 - num ** 2))))
    return e1, e2, g12, g23, nu12, nu23

## 7. Along the fibers the models agree; across them they differ

In [ ]:
models = {"ROM": rule_of_mixtures, "Chamis": chamis, "CCA": cca, "MT": mori_tanaka}
names = ("E1", "E2", "G12", "G23", "ν12", "ν23")
print(f"{'':6}" + "".join(f"{name:>9}" for name in models))
results = {name: model(fiber, em, num, vf) for name, model in models.items()}
for j, prop in enumerate(names):
    print(f"{prop:6}" + "".join(f"{results[name][j]:9.3f}" for name in models))

In [ ]:
vfs = np.linspace(0.0, 0.9, 86)
fig, ax = plt.subplots(figsize=(6, 4))
for (name, model), color in zip(models.items(), ("C0", "C1", "C2", "C3")):
    props = np.array([model(fiber, em, num, v) for v in vfs])
    ax.plot(vfs, props[:, 0], color=color, label=f"E1 {name}")
    ax.plot(vfs, props[:, 1], color=color, linestyle="--", label=f"E2 {name}")
ax.set_xlabel("Vf")
ax.set_ylabel("E (GPa)")
ax.legend(ncol=2, fontsize=8)
plt.show()

## 8. Stiffness for its weight

The specific stiffness E/ρ compares materials for the same weight.
Steel: E = 206.8 GPa, ρ = 7.80 g/cm³ (Kaw).

In [ ]:
e1, e2 = cca(fiber, em, num, vf)[:2]
print(f"composite: E1/ρ = {e1 / rho_c:.1f},  E2/ρ = {e2 / rho_c:.1f} GPa per g/cm³")
print(f"steel:     E/ρ  = {206.8 / 7.80:.1f} GPa per g/cm³")

## Your turn

1. Set `vf = 0.3` in step 1 and run all cells: E1 halves, but E2 drops much less.
2. Use an aluminum matrix: `em, num = 71.0, 0.30` and `rho_m = 2.70`.
   E2 grows about 4 times; E1 only a little.
3. Try a glass fiber: `fiber = glass` (from step 3) with `rho_f = 2.54`.